# SFT Training: nanoVLM for MiniGrid navigation

Trains the ModalityProjector of `lusxvr/nanoVLM-460M-8k` on expert MiniGrid trajectories.
Vision encoder and language decoder are frozen.

In [ ]:
import sys
sys.path.append("nanoVLM")

import json
import os
import math
import time

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

from models.vision_language_model import VisionLanguageModel
from data.processors import get_image_processor
from eval_utils import make_image_string, parse_action

print("Imports OK")

In [ ]:
# ---------- Config ----------
JSONL_PATH = "data/sft_dataset_crop.jsonl"
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 1e-4
EPOCHS = 20
MAX_LENGTH = 2048  # max sequence length (padded)

# Curriculum learning: train on easy (goal visible) first, then all data
CURRICULUM = True
EASY_EPOCHS = 10  # epochs for stage 1 (goal-visible only)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ---------- Load model ----------
print("Loading nanoVLM-460M-8k...")
model = VisionLanguageModel.from_pretrained("lusxvr/nanoVLM-460M-8k")
model.to(device=device)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

# Freeze vision encoder
for p in model.vision_encoder.parameters():
    p.requires_grad = False
print(f"Vision encoder frozen: {sum(p.numel() for p in model.vision_encoder.parameters()):,} params")

# Freeze language decoder
for p in model.decoder.parameters():
    p.requires_grad = False
print(f"Language decoder frozen: {sum(p.numel() for p in model.decoder.parameters()):,} params")

# Train only ModalityProjector
for p in model.MP.parameters():
    p.requires_grad = True
trainable = sum(p.numel() for p in model.MP.parameters())
print(f"ModalityProjector trainable: {trainable:,} params")

tokenizer = model.tokenizer
image_processor = get_image_processor(
    max_img_size=model.cfg.max_img_size,
    splitted_image_size=model.cfg.vit_img_size,
    resize_to_max_side_len=model.cfg.resize_to_max_side_len
)

In [ ]:
# (make_image_string is imported from eval_utils)

In [ ]:
# ---------- Dataset ----------
class MiniGridVLMDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, image_processor, mp_len, filter_visible=None):
        with open(jsonl_path) as f:
            all_data = [json.loads(line) for line in f]
        if filter_visible is not None:
            self.data = [d for d in all_data if d.get("goal_visible", False) == filter_visible]
            print(f"  Filtered to {len(self.data)}/{len(all_data)} samples (goal_visible={filter_visible})")
        else:
            self.data = all_data
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.mp_len = mp_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # ---- Image ----
        image = Image.open(item["images"][0]).convert("RGB")
        # processed shape: [num_patches, 3, 512, 512]
        processed_image, (n_h, n_w) = self.image_processor(image)

        # ---- Text ----
        prompt = item["texts"][0]["user"]
        answer = item["texts"][0]["assistant"]

        image_string = make_image_string(self.tokenizer, n_h, n_w, self.mp_len)

        messages = [
            {"role": "user", "content": image_string + prompt},
            {"role": "assistant", "content": answer},
        ]

        # ---- Tokenize ----
        out = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_special_tokens=False, return_dict=True
        )
        full_ids = torch.tensor(out["input_ids"], dtype=torch.long)
        attention_mask = torch.ones_like(full_ids)

        # ---- Labels: -100 for user tokens ----
        out_user = self.tokenizer.apply_chat_template(
            messages[:1], tokenize=True, add_special_tokens=False, return_dict=True
        )
        user_ids = out_user["input_ids"]
        labels = full_ids.clone()
        labels[:len(user_ids)] = -100
        # Shift labels for causal LM (predict next token)
        labels = labels.roll(-1)
        labels[-1] = -100

        return {
            "input_ids": full_ids,
            "labels": labels,
            "attention_mask": attention_mask,
            "images": processed_image,  # [num_patches, 3, 512, 512]
        }

In [ ]:
# ---------- Collator (left-padding, same as nanoVLM VQACollator) ----------
class MiniGridCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        batch = [b for b in batch if b is not None]
        if not batch:
            return None

        # Keep images as a list (model._process_images concatenates along dim=0)
        images = [b["images"] for b in batch]

        input_ids = [b["input_ids"] for b in batch]
        labels = [b["labels"] for b in batch]
        attention_mask = [b["attention_mask"] for b in batch]

        # Left-pad to max_length
        pad_id = self.tokenizer.pad_token_id
        padded_ids, padded_labels, padded_mask = [], [], []
        for ids, lbl, am in zip(input_ids, labels, attention_mask):
            length = len(ids)
            if length > self.max_length:
                # Truncate from the right (keep the prompt answer part)
                ids = ids[:self.max_length]
                lbl = lbl[:self.max_length]
                am = am[:self.max_length]
                length = self.max_length
            pad_len = self.max_length - length
            padded_ids.append(F.pad(ids, (pad_len, 0), value=pad_id))
            padded_labels.append(F.pad(lbl, (pad_len, 0), value=-100))
            padded_mask.append(F.pad(am, (pad_len, 0), value=0))

        return {
            "input_ids": torch.stack(padded_ids),
            "labels": torch.stack(padded_labels),
            "attention_mask": torch.stack(padded_mask),
            "images": images,
        }

In [ ]:
# ---------- Create dataloader (stage 1: goal visible) ----------
if CURRICULUM:
    dataset = MiniGridVLMDataset(
        JSONL_PATH, tokenizer, image_processor, model.cfg.mp_image_token_length,
        filter_visible=True,
    )
else:
    dataset = MiniGridVLMDataset(
        JSONL_PATH, tokenizer, image_processor, model.cfg.mp_image_token_length,
    )
collator = MiniGridCollator(tokenizer, MAX_LENGTH)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
    num_workers=0, pin_memory=False
)
print(f"Dataset size: {len(dataset)} samples, {len(dataloader)} batches")

In [ ]:
# ---------- Optimizer ----------
trainable_params = [p for p in model.MP.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR)
print(f"Optimizing {sum(p.numel() for p in trainable_params):,} parameters")

In [ ]:
# ---------- Training loop (curriculum: easy -> all) ----------
model.train()
global_step = 0

if CURRICULUM:
    stages = [(1, EASY_EPOCHS, True, "Easy (goal visible)")]
    if EASY_EPOCHS < EPOCHS:
        stages.append((EASY_EPOCHS + 1, EPOCHS, False, "All data"))
else:
    stages = [(1, EPOCHS, None, "All data")]

for start_epoch, end_epoch, filter_visible, stage_name in stages:
    if CURRICULUM:
        dataset = MiniGridVLMDataset(
            JSONL_PATH, tokenizer, image_processor, model.cfg.mp_image_token_length,
            filter_visible=filter_visible,
        )
        dataloader = DataLoader(
            dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
            num_workers=0, pin_memory=False
        )
        print(f"Stage: {stage_name}, {len(dataset)} samples")

    for epoch in range(start_epoch, end_epoch + 1):
        epoch_loss = 0.0
        num_batches = 0
        optimizer.zero_grad()
        start = time.time()

        pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}")
        for batch in pbar:
            if batch is None:
                continue

            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            images = batch["images"]

            # Forward with mixed precision
            with torch.autocast(device_type=device, dtype=torch.float16):
                _, loss = model(
                    input_ids, images,
                    attention_mask=attention_mask,
                    targets=labels,
                )

            loss = loss / GRAD_ACCUM
            loss.backward()

            if (num_batches + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1

            epoch_loss += loss.item() * GRAD_ACCUM
            num_batches += 1
            pbar.set_postfix({"loss": f"{loss.item() * GRAD_ACCUM:.4f}"})

        avg_loss = epoch_loss / num_batches
        elapsed = time.time() - start
        print(f"Epoch {epoch} avg loss: {avg_loss:.4f}, time: {elapsed:.1f}s")

        # Save checkpoint every 5 epochs
        if epoch % 5 == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"sft_epoch_{epoch}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"Saved {ckpt_path}")

print("Training complete!")

In [ ]:
# Save final model
final_path = os.path.join(CHECKPOINT_DIR, "sft_model.pt")
torch.save(model.state_dict(), final_path)
print(f"Final model saved to {final_path}")

In [ ]:
# ---------- Evaluate final checkpoint ----------
from env_utils import create_env, get_agent_view
from expert import bfs_path


@torch.inference_mode()
def evaluate(model, tokenizer, image_processor, num_episodes=10, max_steps=100):
    env = create_env()
    mp_len = model.cfg.mp_image_token_length
    prompt = (
        "What is the next action to reach the green goal? "
        "Choose from: turn left, turn right, move forward."
    )

    successes = 0
    total_steps = 0
    oracle_steps = 0

    for ep in range(num_episodes):
        env.reset()
        oracle = bfs_path(env)
        oracle_len = len(oracle) if oracle else 0
        oracle_steps += oracle_len

        done = False
        truncated = False
        step = 0

        while not (done or truncated) and step < max_steps:
            img = get_agent_view(env)
            processed_img, (n_h, n_w) = image_processor(img)

            image_string = make_image_string(tokenizer, n_h, n_w, mp_len)
            messages = [
                {"role": "user", "content": image_string + prompt},
            ]

            input_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_special_tokens=False
            )
            input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)
            attention_mask = torch.ones_like(input_ids)

            generated_ids = model.generate(
                input_ids,
                images=[processed_img.to(device=device)],
                attention_mask=attention_mask,
                max_new_tokens=10,
                top_k=1,
                greedy=True,
            )

            generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            action = parse_action(generated_text)

            if action is None:
                action = 2

            _, reward, done, truncated, _ = env.step(action)
            step += 1

        if done:
            successes += 1

        total_steps += step
        print(
            f"Ep {ep + 1}/{num_episodes}: "
            f"{'SUCCESS' if done else 'FAIL'} "
            f"({step}/{oracle_len} steps)"
        )

    success_rate = successes / num_episodes * 100
    avg_steps = total_steps / num_episodes
    avg_oracle = oracle_steps / num_episodes
    return success_rate, avg_steps, avg_oracle


final_path = os.path.join(CHECKPOINT_DIR, "sft_model.pt")
model.load_state_dict(torch.load(final_path, map_location=device), strict=False)
model.eval()
sr, avg_s, avg_o = evaluate(model, tokenizer, image_processor, num_episodes=10)
print(f"\n=== Final model: Success rate: {sr:.1f}%, Avg steps: {avg_s:.1f}, Oracle: {avg_o:.1f} ===")